In [ ]:
import pandas as pd

# 1. 열 이름 읽기 (Windows 줄바꿈 \r\n 대비 strip 처리)
with open("Faults27x7_var", "r") as f:
    cols = [line.strip() for line in f if line.strip()]

# 2. 데이터 읽기 (탭/공백 구분, 헤더 없음)
data = pd.read_csv("Faults.NNA", sep=r"\s+", header=None, names=cols)

# 3. 확인
print(data.shape)          # (1941, 34)
print(data.isna().sum().sum())  # 0

# 4. CSV로 저장
data.to_csv("steel_plates_faults.csv", index=False, encoding="utf-8-sig")

df = pd.read_csv("steel_plates_faults.csv")
display(df.head())
display(df.shape)         
display(df.isna().sum().sum())
display(df.info())

(1941, 34)
0


,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Orientation_Index,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults
0,42,50,270900,270944,267,17,44,24220,76,108,...,0.8182,-0.2913,0.5822,1,0,0,0,0,0,0
1,645,651,2538079,2538108,108,10,30,11397,84,123,...,0.7931,-0.1756,0.2984,1,0,0,0,0,0,0
2,829,835,1553913,1553931,71,8,19,7972,99,125,...,0.6667,-0.1228,0.2150,1,0,0,0,0,0,0
3,853,860,369370,369415,176,13,45,18996,99,126,...,0.8444,-0.1568,0.5212,1,0,0,0,0,0,0
4,1289,1306,498078,498335,2409,60,260,246930,37,126,...,0.9338,-0.1992,1.0000,1,0,0,0,0,0,0


(1941, 34)

np.int64(0)

<class 'pandas.DataFrame'>
RangeIndex: 1941 entries, 0 to 1940
Data columns (total 34 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   X_Minimum              1941 non-null   int64  
 1   X_Maximum              1941 non-null   int64  
 2   Y_Minimum              1941 non-null   int64  
 3   Y_Maximum              1941 non-null   int64  
 4   Pixels_Areas           1941 non-null   int64  
 5   X_Perimeter            1941 non-null   int64  
 6   Y_Perimeter            1941 non-null   int64  
 7   Sum_of_Luminosity      1941 non-null   int64  
 8   Minimum_of_Luminosity  1941 non-null   int64  
 9   Maximum_of_Luminosity  1941 non-null   int64  
 10  Length_of_Conveyer     1941 non-null   int64  
 11  TypeOfSteel_A300       1941 non-null   int64  
 12  TypeOfSteel_A400       1941 non-null   int64  
 13  Steel_Plate_Thickness  1941 non-null   int64  
 14  Edges_Index            1941 non-null   float64
 15  Empty_Index    

None

TypeOfSteel
1    1164
0     777
Name: count, dtype: int64

# 전처리

In [11]:
# 파생 변수 확인

"""
강판 결함 데이터: 지수(Index) 변수가 기본 측정값으로 계산된 파생 변수인지 검증
- 계산식으로 직접 값을 만들고, 원본 열과 비교해 일치율과 오차를 출력합니다.
- 원본 값은 소수점 4자리로 반올림되어 있으므로 허용 오차(TOL) 이내면 일치로 봅니다.
"""
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# 1. 데이터 불러오기 (CSV로 변환한 파일 사용)
# ---------------------------------------------------------------
df = pd.read_csv("steel_plates_faults.csv")
# NNA 원본을 바로 쓰려면 아래 두 줄로 대체
# cols = [c.strip() for c in open("Faults27x7_var") if c.strip()]
# df = pd.read_csv("Faults.NNA", sep=r"\s+", header=None, names=cols)

TOL = 1e-3  # 반올림 오차 허용 범위

# ---------------------------------------------------------------
# 2. 계산에 쓸 기본 값
# ---------------------------------------------------------------
dx = df["X_Maximum"] - df["X_Minimum"]          # 결함 가로 길이
dy = df["Y_Maximum"] - df["Y_Minimum"]          # 결함 세로 길이
L = df["Length_of_Conveyer"]                    # 컨베이어(강판 폭) 길이
area = df["Pixels_Areas"]                       # 결함 면적
mean_lum = df["Sum_of_Luminosity"] / area       # 평균 밝기

# ---------------------------------------------------------------
# 3. 계산식으로 파생 변수 직접 생성
# ---------------------------------------------------------------
calc = pd.DataFrame(index=df.index)
calc["Edges_Index"] = 2 * np.minimum(df["X_Minimum"], L - df["X_Maximum"]) / L
calc["Empty_Index"] = 1 - area / (dx * dy)
calc["Square_Index"] = np.minimum(dx, dy) / np.maximum(dx, dy)
calc["Outside_X_Index"] = dx / L
calc["Edges_X_Index"] = dx / df["X_Perimeter"]
calc["Edges_Y_Index"] = dy / df["Y_Perimeter"]
calc["Outside_Global_Index"] = np.select([dy > dx, dy == dx], [1.0, 0.5], default=0.0)
calc["LogOfAreas"] = np.log10(area)
calc["Log_X_Index"] = np.log10(dx)
calc["Log_Y_Index"] = np.log10(dy)
calc["Orientation_Index"] = np.where(dy >= dx, 1 - dx / dy, -(1 - dy / dx))
calc["Luminosity_Index"] = mean_lum / 128 - 1

# ---------------------------------------------------------------
# 4. 원본 값과 비교
# ---------------------------------------------------------------
rows = []
for col in calc.columns:
    diff = (df[col] - calc[col]).abs()
    rows.append({
        "변수": col,
        "일치 행 수": int((diff <= TOL).sum()),
        "전체 행 수": len(df),
        "일치율(%)": round((diff <= TOL).mean() * 100, 2),
        "최대 오차": round(diff.max(), 6),
        "평균 오차": round(diff.mean(), 6),
        "상관계수": round(np.corrcoef(df[col], calc[col])[0, 1], 6),
    })

result = pd.DataFrame(rows)
print("=== 파생 변수 계산식 검증 결과 ===")
print(result.to_string(index=False))

# ---------------------------------------------------------------
# 5. 일치하지 않는 행이 있으면 예시 출력
# ---------------------------------------------------------------
for col in calc.columns:
    diff = (df[col] - calc[col]).abs()
    bad = diff > TOL
    if bad.any():
        print(f"\n[{col}] 불일치 {bad.sum()}행 예시")
        print(pd.DataFrame({"원본": df.loc[bad, col],
                            "계산값": calc.loc[bad, col].round(4),
                            "오차": diff[bad].round(4)}).head())

# ---------------------------------------------------------------
# 6. 강종 더미 변수 중복 여부 확인
# ---------------------------------------------------------------
steel_sum = df["TypeOfSteel_A300"] + df["TypeOfSteel_A400"]
print("\n=== 강종 더미 변수 확인 ===")
print("A300 + A400 값 분포:", steel_sum.value_counts().to_dict())
if (steel_sum == 1).all():
    print("→ 모든 행에서 합이 1: 두 열은 완전히 겹치는 정보이므로 하나만 사용")

# ---------------------------------------------------------------
# 7. 면적 관련 변수 간 상관 (SigmoidOfAreas는 식 미확인, 상관만 확인)
# ---------------------------------------------------------------
print("\n=== 면적 관련 변수 상관계수 (Spearman) ===")
print(df[["Pixels_Areas", "LogOfAreas", "SigmoidOfAreas"]]
      .corr(method="spearman").round(4))

## 면적 관련 변수인 Pixels_Areas, LogOfAreas, SigmoidOfAreas는 높은 상관관계를 보이지만, 각각 원본 면적과 서로 다른 변환 정보를 나타낸다. 
## 따라서 사전에 변수를 제거하기보다는 세 변수를 모두 유지한 상태에서 분석을 진행하고, 
## 변수 제거 전후의 분류 성능을 비교하여 중복 정보가 모델에 미치는 영향을 확인하고자 한다. 
## 이를 통해 변수 간 상관관계뿐 아니라 실제 예측 성능을 고려해 최종 변수를 선정한다.

=== 파생 변수 계산식 검증 결과 ===
                  변수  일치 행 수  전체 행 수  일치율(%)    최대 오차    평균 오차  상관계수
         Edges_Index    1941    1941   100.0 0.000055 0.000024   1.0
         Empty_Index    1941    1941   100.0 0.000055 0.000024   1.0
        Square_Index    1941    1941   100.0 0.000055 0.000022   1.0
     Outside_X_Index    1941    1941   100.0 0.000055 0.000025   1.0
       Edges_X_Index    1941    1941   100.0 0.000055 0.000020   1.0
       Edges_Y_Index    1941    1941   100.0 0.000055 0.000014   1.0
Outside_Global_Index    1941    1941   100.0 0.000000 0.000000   1.0
          LogOfAreas    1941    1941   100.0 0.000055 0.000025   1.0
         Log_X_Index    1941    1941   100.0 0.000054 0.000025   1.0
         Log_Y_Index    1941    1941   100.0 0.000054 0.000025   1.0
   Orientation_Index    1941    1941   100.0 0.000055 0.000022   1.0
    Luminosity_Index    1941    1941   100.0 0.000055 0.000026   1.0

=== 강종 더미 변수 확인 ===
A300 + A400 값 분포: {1: 1941}
→ 모든 행에서 합이 1: 두 열은 완전히 겹치는 정보

In [12]:
# 1. 결함 유형을 하나의 열로 통합
fault_cols = [
    "Pastry", "Z_Scratch", "K_Scatch", "Stains",
    "Dirtiness", "Bumps", "Other_Faults"
]

df1 = df.copy()

df1["Fault"] = df1[fault_cols].idxmax(axis=1)

# 기존 결함 유형 열 제거
df1 = df1.drop(columns=fault_cols)


# 2. 강종을 하나의 열로 통합
# TypeOfSteel_A300: 0이면 A300, 1이면 A400
df1["TypeOfSteel"] = df1["TypeOfSteel_A400"]

# 기존 강종 열 제거
df1 = df1.drop(columns=["TypeOfSteel_A300", "TypeOfSteel_A400"])


# 3. df1 확인
display(df1.head())
print("df1 크기:", df1.shape)


# 4. df1에서 파생변수 제거하여 df2 생성
derived_cols = [
    "Edges_Index",
    "Empty_Index",
    "Square_Index",
    "Outside_X_Index",
    "Edges_X_Index",
    "Edges_Y_Index",
    "Outside_Global_Index",
    "LogOfAreas",
    "Log_X_Index",
    "Log_Y_Index",
    "Orientation_Index",
    "Luminosity_Index"
]

df2 = df1.drop(columns=derived_cols).copy()


# 5. df2 확인
display(df2.head())
print("df2 크기:", df2.shape)

,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Edges_Y_Index,Outside_Global_Index,LogOfAreas,Log_X_Index,Log_Y_Index,Orientation_Index,Luminosity_Index,SigmoidOfAreas,Fault,TypeOfSteel
0,42,50,270900,270944,267,17,44,24220,76,108,...,1.0000,1.0,2.4265,0.9031,1.6435,0.8182,-0.2913,0.5822,Pastry,0
1,645,651,2538079,2538108,108,10,30,11397,84,123,...,0.9667,1.0,2.0334,0.7782,1.4624,0.7931,-0.1756,0.2984,Pastry,0
2,829,835,1553913,1553931,71,8,19,7972,99,125,...,0.9474,1.0,1.8513,0.7782,1.2553,0.6667,-0.1228,0.2150,Pastry,0
3,853,860,369370,369415,176,13,45,18996,99,126,...,1.0000,1.0,2.2455,0.8451,1.6532,0.8444,-0.1568,0.5212,Pastry,1
4,1289,1306,498078,498335,2409,60,260,246930,37,126,...,0.9885,1.0,3.3818,1.2305,2.4099,0.9338,-0.1992,1.0000,Pastry,1


df1 크기: (1941, 27)


,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,Length_of_Conveyer,Steel_Plate_Thickness,SigmoidOfAreas,Fault,TypeOfSteel
0,42,50,270900,270944,267,17,44,24220,76,108,1687,80,0.5822,Pastry,0
1,645,651,2538079,2538108,108,10,30,11397,84,123,1687,80,0.2984,Pastry,0
2,829,835,1553913,1553931,71,8,19,7972,99,125,1623,100,0.2150,Pastry,0
3,853,860,369370,369415,176,13,45,18996,99,126,1353,290,0.5212,Pastry,1
4,1289,1306,498078,498335,2409,60,260,246930,37,126,1353,185,1.0000,Pastry,1


df2 크기: (1941, 15)


In [14]:

import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# 1. 데이터 불러오기 및 변수 구분
# ============================================================
df = pd.read_csv("steel_plates_faults.csv")

# 설명 변수 X: 강판의 결함 특성 27개
X_cols = df.columns[:27].tolist()

# 타깃 변수 Y: 결함 유형 7개
Y_cols = df.columns[27:].tolist()

X = df[X_cols]
Y = df[Y_cols]

n = len(df)

print("데이터 크기:", df.shape)
print("설명 변수 개수:", len(X_cols))
print("타깃 변수 개수:", len(Y_cols))


# ============================================================
# 2. 결측치 현황
# ============================================================
missing_count = df.isna().sum()
missing_ratio = missing_count / n * 100

missing_result = pd.DataFrame({
    "결측 건수": missing_count,
    "결측 비율(%)": missing_ratio.round(2),
    "처리 방향": [
        "결측 없음" if count == 0
        else "결측 원인 확인 후 대체 또는 행 제거 검토"
        for count in missing_count
    ]
})

print("\n[결측치 현황]")
display(missing_result)

print("전체 결측 건수:", int(missing_count.sum()))


# ============================================================
# 3. 이상치 현황
#    기준: IQR (Q1 - 1.5*IQR, Q3 + 1.5*IQR)
#    이진 타깃 및 이진 강종 변수는 제외
# ============================================================
binary_cols = [
    col for col in df.columns
    if df[col].nunique(dropna=True) <= 2
    and set(df[col].dropna().unique()).issubset({0, 1})
]

outlier_cols = [
    col for col in X_cols
    if col not in binary_cols
    and pd.api.types.is_numeric_dtype(df[col])
]

outlier_results = []

for col in outlier_cols:
    series = df[col].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = ((series < lower) | (series > upper)).sum()
    ratio = count / len(series) * 100 if len(series) else 0

    outlier_results.append({
        "컬럼": col,
        "하한": lower,
        "상한": upper,
        "이상치 건수": int(count),
        "이상치 비율(%)": round(ratio, 2),
        "처리 방향": (
            "분포 및 원자료 확인 후 유지 여부 판단"
            if count > 0 else "IQR 기준 이상치 없음"
        )
    })

outlier_result = pd.DataFrame(outlier_results)

print("\n[IQR 기준 이상치 현황]")
display(
    outlier_result.sort_values(
        "이상치 건수", ascending=False
    ).reset_index(drop=True)
)


# ============================================================
# 4. 타깃 분류 및 클래스 분포
# ============================================================
print("\n[설명 변수 X]")
print("컬럼 목록:", X_cols)
print("데이터 타입:")
display(X.dtypes.rename("데이터 타입").to_frame())

print("\n[타깃 변수 Y]")
target_info = pd.DataFrame({
    "정의": ["해당 결함 유형 존재 여부(0/1)"] * len(Y_cols),
    "데이터 타입": Y.dtypes.astype(str).values,
    "0 건수": [(Y[col] == 0).sum() for col in Y_cols],
    "1 건수": [(Y[col] == 1).sum() for col in Y_cols],
    "1 비율(%)": [
        round((Y[col] == 1).mean() * 100, 2)
        for col in Y_cols
    ]
}, index=Y_cols)

display(target_info)

# 타깃 값이 0/1로만 구성되어 있는지 확인
binary_check = {
    col: set(Y[col].dropna().unique()).issubset({0, 1})
    for col in Y_cols
}

print("타깃 변수의 0/1 구성 여부:", binary_check)

# 한 행에 여러 결함 유형이 동시에 표시되는지 확인
labels_per_row = Y.sum(axis=1)

print("\n[행별 타깃 개수]")
display(labels_per_row.value_counts().sort_index().rename(
    "행 개수"
).to_frame())

print(
    "복수 타깃(2개 이상)이 표시된 행:",
    int((labels_per_row >= 2).sum())
)

print(
    "타깃이 하나도 표시되지 않은 행:",
    int((labels_per_row == 0).sum())
)


# ============================================================
# 5. 데이터 누수 가능성 점검
# ============================================================
print("\n[데이터 누수 점검]")

# 5-1. X 내부 완전 중복 컬럼 확인
duplicate_x_cols = X.columns[X.T.duplicated()].tolist()

print("X 내부 완전 중복 컬럼:", duplicate_x_cols)

# 5-2. 타깃 변수의 값이 0/1인지 확인
invalid_targets = [
    col for col in Y_cols
    if not set(Y[col].dropna().unique()).issubset({0, 1})
]

print("0/1 이외의 값이 있는 타깃:", invalid_targets)

# 5-3. 설명 변수와 타깃 간 절대 상관계수 확인
# 상관관계가 높다는 이유만으로 누수라고 단정하지 않음
corr_with_target = X.join(Y).corr(numeric_only=True).loc[
    X_cols, Y_cols
]

target_corr_result = (
    corr_with_target.abs()
    .stack()
    .rename("절대 상관계수")
    .reset_index()
)

target_corr_result.columns = [
    "설명 변수", "타깃 변수", "절대 상관계수"
]

print("\n[설명 변수와 타깃 간 상관관계 상위 15개]")
display(
    target_corr_result.sort_values(
        "절대 상관계수", ascending=False
    ).head(15).reset_index(drop=True)
)

# 5-4. 행 중복 확인
print("완전 중복 행 수:", int(df.duplicated().sum()))

# 5-5. 타깃 변수별 상호 배타성 확인
# 여러 결함 유형이 동시에 표시될 수 있으므로 별도 확인
print(
    "타깃이 2개 이상인 행 수:",
    int((Y.sum(axis=1) >= 2).sum())
)


# ============================================================
# 6. 개인정보·민감정보 점검
# ============================================================
# 컬럼명 기반의 1차 점검이며, 데이터 내용까지 확정하는 검사는 아님
privacy_keywords = [
    "name", "email", "phone", "address",
    "resident", "ssn", "id_number",
    "birth", "account", "password"
]

privacy_candidates = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in privacy_keywords)
]

print("\n[개인정보·민감정보 점검]")
print("컬럼명 기반 의심 항목:", privacy_candidates)

if privacy_candidates:
    print("조치: 해당 컬럼의 실제 내용을 확인하고 필요 시 삭제 또는 비식별화")
else:
    print("컬럼명 기준 개인정보 의심 항목 없음")
    print("단, 컬럼명만으로 개인정보 미포함을 확정할 수는 없음")


# ============================================================
# 7. 보고서용 핵심 요약
# ============================================================
print("\n" + "=" * 50)
print("[데이터 품질 및 모델링 요약]")
print("=" * 50)

print(f"전체 데이터: {n:,}행 × {df.shape[1]}열")
print(f"설명 변수 X: {len(X_cols)}개")
print(f"타깃 변수 Y: {len(Y_cols)}개")
print(f"전체 결측치: {int(df.isna().sum().sum()):,}개")
print(f"IQR 기준 이상치: {outlier_result['이상치 건수'].sum():,}건")
print(f"완전 중복 행: {int(df.duplicated().sum()):,}개")
print("분류 유형: 다중 레이블 분류")


데이터 크기: (1941, 34)
설명 변수 개수: 27
타깃 변수 개수: 7

[결측치 현황]


,결측 건수,결측 비율(%),처리 방향
X_Minimum,0,0.0,결측 없음
X_Maximum,0,0.0,결측 없음
Y_Minimum,0,0.0,결측 없음
Y_Maximum,0,0.0,결측 없음
Pixels_Areas,0,0.0,결측 없음
X_Perimeter,0,0.0,결측 없음
Y_Perimeter,0,0.0,결측 없음
Sum_of_Luminosity,0,0.0,결측 없음
Minimum_of_Luminosity,0,0.0,결측 없음
Maximum_of_Luminosity,0,0.0,결측 없음


전체 결측 건수: 0

[IQR 기준 이상치 현황]


,컬럼,하한,상한,이상치 건수,이상치 비율(%),처리 방향
0,Sum_of_Luminosity,-1.007115e+05,1.932445e+05,399,20.56,분포 및 원자료 확인 후 유지 여부 판단
1,Pixels_Areas,-1.023000e+03,1.929000e+03,395,20.35,분포 및 원자료 확인 후 유지 여부 판단
2,Outside_X_Index,-1.875000e-02,4.885000e-02,370,19.06,분포 및 원자료 확인 후 유지 여부 판단
3,X_Perimeter,-8.850000e+01,1.875000e+02,352,18.13,분포 및 원자료 확인 후 유지 여부 판단
4,Steel_Plate_Thickness,-2.000000e+01,1.400000e+02,240,12.36,분포 및 원자료 확인 후 유지 여부 판단
5,Y_Perimeter,-9.200000e+01,1.880000e+02,179,9.22,분포 및 원자료 확인 후 유지 여부 판단
6,Maximum_of_Luminosity,1.000000e+02,1.640000e+02,146,7.52,분포 및 원자료 확인 후 유지 여부 판단
7,Luminosity_Index,-3.876000e-01,1.260000e-01,134,6.90,분포 및 원자료 확인 후 유지 여부 판단
8,Y_Minimum,-2.096477e+06,4.750803e+06,81,4.17,분포 및 원자료 확인 후 유지 여부 판단
9,Y_Maximum,-2.096424e+06,4.750788e+06,81,4.17,분포 및 원자료 확인 후 유지 여부 판단



[설명 변수 X]
컬럼 목록: ['X_Minimum', 'X_Maximum', 'Y_Minimum', 'Y_Maximum', 'Pixels_Areas', 'X_Perimeter', 'Y_Perimeter', 'Sum_of_Luminosity', 'Minimum_of_Luminosity', 'Maximum_of_Luminosity', 'Length_of_Conveyer', 'TypeOfSteel_A300', 'TypeOfSteel_A400', 'Steel_Plate_Thickness', 'Edges_Index', 'Empty_Index', 'Square_Index', 'Outside_X_Index', 'Edges_X_Index', 'Edges_Y_Index', 'Outside_Global_Index', 'LogOfAreas', 'Log_X_Index', 'Log_Y_Index', 'Orientation_Index', 'Luminosity_Index', 'SigmoidOfAreas']
데이터 타입:


...

타깃 변수의 0/1 구성 여부: {'Pastry': True, 'Z_Scratch': True, 'K_Scatch': True, 'Stains': True, 'Dirtiness': True, 'Bumps': True, 'Other_Faults': True}

[행별 타깃 개수]


,행 개수
1,1941


복수 타깃(2개 이상)이 표시된 행: 0
타깃이 하나도 표시되지 않은 행: 0

[데이터 누수 점검]
X 내부 완전 중복 컬럼: []
0/1 이외의 값이 있는 타깃: []

[설명 변수와 타깃 간 상관관계 상위 15개]


,설명 변수,타깃 변수,절대 상관계수
0,Log_X_Index,K_Scatch,0.735860
1,LogOfAreas,K_Scatch,0.704531
2,Outside_X_Index,K_Scatch,0.631370
3,Sum_of_Luminosity,K_Scatch,0.616950
4,Edges_Y_Index,K_Scatch,0.614341
5,Pixels_Areas,K_Scatch,0.556846
6,SigmoidOfAreas,K_Scatch,0.488878
7,Log_Y_Index,K_Scatch,0.481853
8,Minimum_of_Luminosity,K_Scatch,0.461000
9,X_Perimeter,K_Scatch,0.455003


완전 중복 행 수: 0
타깃이 2개 이상인 행 수: 0

[개인정보·민감정보 점검]
컬럼명 기반 의심 항목: []
컬럼명 기준 개인정보 의심 항목 없음
단, 컬럼명만으로 개인정보 미포함을 확정할 수는 없음

[데이터 품질 및 모델링 요약]
전체 데이터: 1,941행 × 34열
설명 변수 X: 27개
타깃 변수 Y: 7개
전체 결측치: 0개
IQR 기준 이상치: 2,461건
완전 중복 행: 0개
분류 유형: 다중 레이블 분류
